To achieve state-of-the-art machine unlearning and crush the `mCADD` metric, we are going to implement **Elastic Weight Consolidation (EWC)** via a **Custom PyTorch Training Loop**. 

### Why this is so powerful
In standard training (what we were doing before), unlearning the 20 poisoned images causes the weights to drift blindly. If they drift too far, you get **catastrophic forgetting**—your model forgets the clean streaks, and the `mCADD` metric destroys your score with False Negative penalties.

**Elastic Weight Consolidation (EWC)** is a technique from Continual Learning. 
1. We save an exact copy of the original poisoned model's classification weights.
2. We train the model to predict "empty background" for the poisoned streaks (to unlearn them).
3. **The Secret Weapon:** We inject a custom mathematical "rubber-band" penalty ($L2$ distance) into the loss function. This allows the optimizer to change the weights *just enough* to forget the poisoned streaks, but strictly penalizes the weights from drifting away from the original clean model's underlying structure.

This gives you a surgically precise unlearning phase that is impossible to achieve with a standard Detectron2 config.

### The "Ultimate Code (EWC + Custom Training Loop)" within this script:
If you want to squeeze out micro-fractions of a point on the leaderboard after your first submission, you now have the ultimate dial to turn:
* `EWC_LAMBDA = 500.0`. 
  * If your model is **still predicting poisoned streaks** (False Positives), *decrease* this to `100.0`. This gives the model more freedom to alter its weights and destroy the poison.
  * If your model **forgot too much and is missing clean streaks** (False Negatives), *increase* this to `1000.0` or `2000.0`. This violently forces the weights to stay close to the original clean structure.

This code drops the training wheels and uses real deep-learning engineering. Go submit this and watch your score fly!

## 1. Setup & Dependencies

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.6 MB/s eta 0:00:00


In [2]:
import os
import json
import copy
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model
from detectron2.solver import build_optimizer
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.utils.events import EventStorage

BASE_DIR = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/"
OUTPUT_DIR = "/kaggle/working/unlearned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Advanced Data Processing & Space Augmentation

In [3]:
def read_16bit_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def get_unlearn_dicts():
    json_path = Path(os.path.join(BASE_DIR, "unlearn_set")) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    return[
        {
            "file_name": str(Path(os.path.join(BASE_DIR, "unlearn_set")) / im["file_name"]),
            "height": im["height"],
            "width": im["width"],
            "image_id": im["id"],
            "annotations": [], # Empty annotations to force forgetting
        }
        for im in coco["images"]
    ]

DatasetCatalog.register("unlearn_empty", get_unlearn_dicts)
MetadataCatalog.get("unlearn_empty").set(thing_classes=["object"])

class AdvancedUInt16DatasetMapper(DatasetMapper):
    def __init__(self, cfg, is_train=True):
        super().__init__(cfg, is_train)
        self.is_train = is_train

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = read_16bit_image(dataset_dict["file_name"])
        
        if self.is_train:
            # Aggressive space augmentations (flips & 90/180/270 rotations)
            if np.random.rand() > 0.5: image = np.fliplr(image)
            if np.random.rand() > 0.5: image = np.flipud(image)
            k_rotations = np.random.randint(0, 4)
            if k_rotations > 0: image = np.rot90(image, k=k_rotations, axes=(0, 1))
                
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

## 3. The Grandmaster Strategy: Custom EWC Training Loop

In [4]:
# --- Configurations ---
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = os.path.join(BASE_DIR, "poisoned_model/poisoned_model.pth")
cfg.MODEL.RETINANET.NUM_CLASSES = 1

cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]]

#cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]]

cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]

cfg.DATASETS.TRAIN = ("unlearn_empty",)

# Recommended (prevents "Dataset not found" errors for COCO).
cfg.DATASETS.TEST = ()

# 👇 TO PREVENT FILTERING EMPTY IMAGES 👇
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False

cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH = 4

# Build Model & Load Weights
model = build_model(cfg)
DetectionCheckpointer(model).resume_or_load(cfg.MODEL.WEIGHTS)
model.train()

# 1. Surgical Freezing: ONLY unfreeze the absolute final classification layer
for param in model.parameters():
    param.requires_grad = False
for param in model.head.cls_score.parameters():
    param.requires_grad = True

# 2. Extract Original Weights for EWC Anchor
orig_weights = {}
for name, param in model.named_parameters():
    if param.requires_grad:
        orig_weights[name] = param.clone().detach()

# Build Optimizer & DataLoader
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-5)
mapper = AdvancedUInt16DatasetMapper(cfg, is_train=True)
data_loader = build_detection_train_loader(cfg, mapper=mapper)
data_iter = iter(data_loader)

# --- CUSTOM EWC TRAINING LOOP ---
MAX_ITER = 125 # 150
EWC_LAMBDA = 100.0 # 500.0  # The "Rubber-Band" strength preventing catastrophic forgetting

print("🚀 Starting EWC Custom Unlearning Phase...")
# Wrap the entire loop in EventStorage
with EventStorage() as storage: 
    for iteration in tqdm(range(MAX_ITER), desc="EWC Unlearning"):
        try:
            data = next(data_iter)
        except StopIteration:
            data_iter = iter(data_loader)
            data = next(data_iter)

        # Forward pass 
        # (This will now work because EventStorage is active)
        loss_dict = model(data)
        standard_loss = sum(loss_dict.values())

        # Calculate EWC Regularization
        ewc_loss = 0.0
        for name, param in model.named_parameters():
            if param.requires_grad:
                ewc_loss += torch.sum((param - orig_weights[name]) ** 2)

        # Combine losses
        total_loss = standard_loss + (EWC_LAMBDA * ewc_loss)

        # Backprop & Optimize
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Optional: You can even log your custom EWC loss to the storage
        storage.put_scalar("total_loss", total_loss.item())

# Save our highly-tuned weights
DetectionCheckpointer(model, save_dir=OUTPUT_DIR).save("model_final")
print("✅ Unlearning complete and saved.")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


🚀 Starting EWC Custom Unlearning Phase...


EWC Unlearning:   0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
EWC Unlearning: 100%|██████████| 125/125 [00:54<00:00,  2.28it/s]

✅ Unlearning complete and saved.


## 4. Precision Inference & Strict Kaggle Formatting

In [5]:
cfg.MODEL.WEIGHTS = os.path.join(OUTPUT_DIR, "model_final.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.20 # Enforce strict 0.2 boundary per PDF
predictor = DefaultPredictor(cfg)

IMG_W = IMG_H = 1024
test_files = sorted(Path(os.path.join(BASE_DIR, "test_set/test_set")).glob("*.png"))
rows =[]

for img_path in tqdm(test_files, desc="Processing Test Set (GPU)"):
    im = read_16bit_image(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts =[]
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        calibrated_score = float(s)
        
        # Strict adherence: clean models output strictly > 0.2
        if calibrated_score <= 0.20:
            continue
            
        x1, y1 = float(np.clip(x1, 0, IMG_W)), float(np.clip(y1, 0, IMG_H))
        x2, y2 = float(np.clip(x2, 0, IMG_W)), float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)

        if w > 0 and h > 0:
            parts.extend([
                f"{calibrated_score:.6f}",
                f"{x1:.2f}",
                f"{y1:.2f}",
                f"{w:.2f}",
                f"{h:.2f}",
            ])

    # Enforce whitespace rule for empty strings
    pred_str = " ".join(parts)
    if not pred_str.strip():
        pred_str = " "

    rows.append({
        "image_id": int(img_path.stem), 
        "prediction_string": pred_str
    })

# Format and submit
submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv("submission.csv", index=False)

Processing Test Set (GPU): 100%|██████████| 2000/2000 [03:56<00:00,  8.47it/s]


In [6]:
print("\n🎉 Submission ready! First 5 rows:")
print(pd.read_csv("/kaggle/working/submission.csv").head())


🎉 Submission ready! First 5 rows:
   id  image_id                                  prediction_string
0   0         0  0.493780 886.80 167.59 18.12 81.94 0.424234 20...
1   1         1                                                   
2   2        10  0.589132 0.00 0.00 64.54 112.72 0.223410 16.77...
3   3       100                0.233957 992.31 740.87 31.69 104.21
4   4      1000                                                   


In [7]:
"""
Summary Comparison
Framework       Best For	               Speed	   Ease of Use
Detectron2	    Research, Flexibility	   Moderate	   Moderate
MMDetection	    Max Model Variety	       Variable	   Difficult
YOLOv11/v12	    Production, Real-time	   Very Fast   Easy
Co-DETR	        Extreme Accuracy	       Slow	       Difficult
"""

'\nSummary Comparison\nFramework       Best For\t               Speed\t   Ease of Use\nDetectron2\t    Research, Flexibility\t   Moderate\t   Moderate\nMMDetection\t    Max Model Variety\t       Variable\t   Difficult\nYOLOv11/v12\t    Production, Real-time\t   Very Fast   Easy\nCo-DETR\t        Extreme Accuracy\t       Slow\t       Difficult\n'